# Dice Detector 🎲 — Kaggle Training (Multi-Head, Option A)

Train a **single-backbone, dual-head `yolo11n`** detector
One model, one forward pass — each detected box yields two independent predictions:

- **dice type** — 7 classes: `D4, D6, D8, D10, D12, D20, D100`
- **die value** — 21 value-classes (`0..20`)


## 1. Setup


In [ ]:
!pip install -q ultralytics pyyaml pillow opencv-python-headless


In [ ]:
from __future__ import annotations

import math
from copy import copy
from types import MethodType
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.nn.modules.conv import Conv, DWConv
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import v8DetectionLoss
import ultralytics.data.dataset as _ds_mod
from ultralytics.models import yolo
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.utils import DEFAULT_CFG, RANK


# --- labels.py ---
import json
from pathlib import Path

DICE_TYPE_TO_CLASS: dict[str, int] = {
    "D4": 0,
    "D6": 1,
    "D8": 2,
    "D10": 3,
    "D12": 4,
    "D20": 5,
    "D100": 6,
}
CLASS_TO_DICE_TYPE: dict[int, str] = {v: k for k, v in DICE_TYPE_TO_CLASS.items()}

NUM_TYPE_CLASSES = 7
NUM_VALUE_CLASSES = 21

# Encoding base. Must be strictly greater than the largest value-class so that
# combined ids are uniquely decodable. 100 leaves plenty of headroom and keeps
# the ids human-readable (e.g. 504 -> type 5 (D20), value-class 4).
VALUE_STRIDE = 100


def value_to_class(dice_type: str, value: int) -> int:
    if dice_type == "D100":
        return int(value) // 10
    return int(value)


def class_to_value(dice_type: str, value_class: int) -> int:
    if dice_type == "D100":
        return int(value_class) * 10
    return int(value_class)


def encode_combined(type_id: int, value_class: int) -> int:
    return int(type_id) * VALUE_STRIDE + int(value_class)


def decode_combined(combined: int) -> tuple[int, int]:
    combined = int(combined)
    return combined // VALUE_STRIDE, combined % VALUE_STRIDE


def convert_annotation(ann_path: Path) -> list[str]:
    with open(ann_path) as f:
        data = json.load(f)

    img_w = data["image_width"]
    img_h = data["image_height"]
    lines: list[str] = []

    for die in data["dice"]:
        type_id = DICE_TYPE_TO_CLASS.get(die["dice_type"])
        if type_id is None:
            continue

        value = die.get("value")
        if value is None:
            continue
        value_class = value_to_class(die["dice_type"], value)
        if not (0 <= value_class < NUM_VALUE_CLASSES):
            continue
        combined = encode_combined(type_id, value_class)

        bbox = die["bbox"]
        x_center = (bbox["x"] + bbox["width"] / 2) / img_w
        y_center = (bbox["y"] + bbox["height"] / 2) / img_h
        width = bbox["width"] / img_w
        height = bbox["height"] / img_h

        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        width = max(0.001, min(1.0, width))
        height = max(0.001, min(1.0, height))

        lines.append(f"{combined} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return lines


def convert_annotations_dir(data_dir: Path) -> int:
    data_dir = Path(data_dir)
    ann_dir = data_dir / "annotations"
    labels_dir = data_dir / "labels"
    if not ann_dir.exists():
        raise FileNotFoundError(f"annotations directory not found: {ann_dir}")
    labels_dir.mkdir(parents=True, exist_ok=True)

    converted = 0
    for ann_path in sorted(ann_dir.glob("*.json")):
        lines = convert_annotation(ann_path)
        label_path = labels_dir / f"{ann_path.stem}.txt"
        label_path.write_text("\n".join(lines) + "\n" if lines else "")
        converted += 1
    return converted


# --- head.py ---
import math

import torch
import torch.nn as nn

from ultralytics.nn.modules.conv import Conv, DWConv
from ultralytics.nn.modules.head import Detect


def _build_value_branch(ch: tuple[int, ...], nv: int, legacy: bool) -> nn.ModuleList:
    c = max(ch[0], min(nv, 100))
    if legacy:
        return nn.ModuleList(
            nn.Sequential(Conv(x, c, 3), Conv(c, c, 3), nn.Conv2d(c, nv, 1)) for x in ch
        )
    return nn.ModuleList(
        nn.Sequential(
            nn.Sequential(DWConv(x, x, 3), Conv(x, c, 1)),
            nn.Sequential(DWConv(c, c, 3), Conv(c, c, 1)),
            nn.Conv2d(c, nv, 1),
        )
        for x in ch
    )


class DetectValue(Detect):
    """Legacy Detect head augmented with an independent value-classification head

    Attributes:
        nv (int): Number of value classes
        cv_val (nn.ModuleList): Per-level value-classification branch
        export_value (bool): When True, ``_inference`` appends the value scores
    """

    export_value: bool = False

    def attach_value_head(self, nv: int) -> "DetectValue":
        """Build and register the value branch on an existing (reclassed) head """
        self.nv = int(nv)
        ch = tuple(seq[0].conv.in_channels for seq in self.cv2)  # input channels per level
        legacy = bool(getattr(self, "legacy", False))
        device = self.cv2[0][-1].weight.device
        dtype = self.cv2[0][-1].weight.dtype

        self.cv_val = _build_value_branch(ch, self.nv, legacy).to(device=device, dtype=dtype)
        self._init_value_bias()
        return self

    def _init_value_bias(self) -> None:
        for i, b in enumerate(self.cv_val):
            stride = self.stride[i] if len(self.stride) > i else self.stride[-1]
            b[-1].bias.data[: self.nv] = math.log(5 / self.nv / (640 / float(stride)) ** 2)

    @property
    def one2many(self) -> dict:
        d = dict(box_head=self.cv2, cls_head=self.cv3)
        if hasattr(self, "cv_val"):
            d["val_head"] = self.cv_val
        return d

    def forward_head(
        self,
        x: list[torch.Tensor],
        box_head: nn.Module = None,
        cls_head: nn.Module = None,
        val_head: nn.Module = None,
    ) -> dict[str, torch.Tensor]:
        """Return box, type-class and value-class predictions for each level"""
        preds = super().forward_head(x, box_head, cls_head)
        if val_head is not None and preds:
            bs = x[0].shape[0]
            preds["value_scores"] = torch.cat(
                [val_head[i](x[i]).view(bs, self.nv, -1) for i in range(self.nl)], dim=-1
            )
        return preds

    def _inference(self, x: dict[str, torch.Tensor]) -> torch.Tensor:
        """Append value-class scores so they ride through NMS when exporting"""
        preds = super()._inference(x)
        if self.export_value and "value_scores" in x:
            preds = torch.cat([preds, x["value_scores"].sigmoid()], dim=1)
        return preds



from typing import Any

import torch
import torch.nn.functional as F

from ultralytics.utils.loss import v8DetectionLoss



class DetectionValueLoss(v8DetectionLoss):
    """v8 detection loss + auxiliary value-classification loss.

    Returns a 4-component loss vector ``[box, cls, dfl, val]``.
    """

    def __init__(self, model: torch.nn.Module, tal_topk: int = 10, tal_topk2: int | None = None):
        super().__init__(model, tal_topk=tal_topk, tal_topk2=tal_topk2)
        m = model.model[-1]
        self.nv = int(getattr(m, "nv"))
        # Gain for the value loss; reuse the cls gain by default.
        self.value_gain = float(getattr(self.hyp, "cls", 0.5))

    def _gt_values(self, batch: dict[str, Any], value_cls: torch.Tensor, batch_size: int) -> torch.Tensor:
        """Build per-image value targets aligned with the assigner's GT indexing

        Returns a ``(batch_size, max_instances)`` float tensor whose ordering
        matches ``target_gt_idx`` from :class:`v8DetectionLoss`'s assigner
        """
        n = value_cls.shape[0]
        targets = torch.cat(
            (
                batch["batch_idx"].view(-1, 1).to(self.device),
                value_cls.view(-1, 1).float(),
                torch.zeros((n, 4), device=self.device),
            ),
            dim=1,
        )
        out = self.preprocess(targets, batch_size, scale_tensor=torch.ones(4, device=self.device))
        return out[..., 0]  # (bs, max_inst)

    def loss(self, preds: dict[str, torch.Tensor], batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
        """Compute box, type-class, DFL and value losses."""
        batch_size = preds["boxes"].shape[0]

        # Decode combined ground-truth ids into type and value classes
        combined = batch["cls"].view(-1).to(self.device)
        type_cls = (combined // VALUE_STRIDE).view(-1, 1)
        value_cls = (combined % VALUE_STRIDE).view(-1, 1)

        # Standard detector loss uses the type class as the detection label.
        batch_type = dict(batch)
        batch_type["cls"] = type_cls

        (fg_mask, target_gt_idx, _, _, _), det_loss, _ = self.get_assigned_targets_and_loss(preds, batch_type)

        loss = torch.zeros(4, device=self.device)  # box, cls, dfl, val
        loss[0], loss[1], loss[2] = det_loss[0], det_loss[1], det_loss[2]

        if "value_scores" in preds and fg_mask.any():
            value_scores = preds["value_scores"].permute(0, 2, 1).contiguous()  # (bs, A, nv)
            gt_values = self._gt_values(batch, value_cls, batch_size)  # (bs, max_inst)
            target_values = gt_values.gather(1, target_gt_idx)  # (bs, A)

            vs = value_scores[fg_mask]  # (Nfg, nv)
            vt = target_values[fg_mask].long().clamp_(0, self.nv - 1)  # (Nfg,)
            loss[3] = F.cross_entropy(vs, vt, reduction="mean") * self.value_gain

        return loss * batch_size, loss.detach()


from types import MethodType

import torch

from ultralytics.nn.tasks import DetectionModel



def _upgrade_head(model: DetectionModel, nv: int) -> None:
    """Reclass the model's Detect head to DetectValue and add the value branch."""
    head = model.model[-1]
    head.__class__ = DetectValue
    head.attach_value_head(nv)


def _patch_criterion(model: DetectionModel) -> None:
    """Make the model build a value-aware (single, non-end2end) criterion."""

    def init_criterion(self):
        return DetectionValueLoss(self)

    model.init_criterion = MethodType(init_criterion, model)
    model.criterion = None  # force lazy rebuild with the patched factory


def build_value_model(
    cfg: str = "yolo11n.yaml",
    nc: int = NUM_TYPE_CLASSES,
    nv: int = NUM_VALUE_CLASSES,
    ch: int = 3,
    weights: str | None = None,
    verbose: bool = True,
) -> DetectionModel:
    """Build a legacy YOLO detection model with an added value-classification head.

    Args:
        cfg: Base model YAML (e.g. ``yolo11n.yaml``).
        nc: Number of dice-type classes (7).
        nv: Number of value classes (21).
        ch: Input channels.
        weights: Optional path to pretrained ``.pt`` weights (backbone + box/type
            head are loaded non-strictly; the value branch stays randomly init).
        verbose: Whether to print model info.

    Returns:
        A ready-to-train :class:`DetectionModel` whose final head is a
        :class:`DetectValue` and whose criterion supervises both heads.
    """
    model = DetectionModel(cfg, nc=nc, ch=ch, verbose=verbose)
    if weights:
        model.load(weights)
    _upgrade_head(model, nv)
    _patch_criterion(model)
    return model


@torch.no_grad()
def predict_values(model: DetectionModel, images, **predict_kwargs):
    head = model.model[-1]
    prev = getattr(head, "export_value", False)
    head.export_value = True
    try:
        return model.predict(images, **predict_kwargs) if hasattr(model, "predict") else model(images)
    finally:
        head.export_value = prev


# --- dataset.py ---
import ultralytics.data.dataset as _ds_mod

_PATCH_FLAG = "_dice_value_verify_patched"


def patch_verify_image_label() -> None:
    """Allow combined class ids (> num type classes) to pass label verification.

    Idempotent: safe to call multiple times.
    """
    if getattr(_ds_mod.verify_image_label, _PATCH_FLAG, False):
        return

    original = _ds_mod.verify_image_label

    def verify_image_label(args):
        a = list(args)
        # args[4] is num_cls; widen it so combined ids are accepted.
        a[4] = max(int(a[4]), 10_000)
        return original(tuple(a))

    setattr(verify_image_label, _PATCH_FLAG, True)
    _ds_mod.verify_image_label = verify_image_label


# --- train.py ---
from copy import copy
from typing import Any

import numpy as np
import torch

from ultralytics.models import yolo
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.utils import DEFAULT_CFG, RANK



class ValueValidator(DetectionValidator):
    def _prepare_batch(self, si: int, batch: dict[str, Any]) -> dict[str, Any]:
        # Decode the combined ground-truth ids to type classes for type/box mAP.
        decoded = dict(batch)
        decoded["cls"] = torch.div(batch["cls"], VALUE_STRIDE, rounding_mode="floor")
        return super()._prepare_batch(si, decoded)


class ValueTrainer(DetectionTrainer):
    nv: int = NUM_VALUE_CLASSES

    def __init__(self, cfg=DEFAULT_CFG, overrides: dict | None = None, _callbacks: dict | None = None):
        patch_verify_image_label()
        super().__init__(cfg, overrides, _callbacks)

    def get_model(self, cfg: str | None = None, weights: Any = None, verbose: bool = True):
        model = build_value_model(
            cfg=cfg,
            nc=self.data["nc"],
            nv=self.nv,
            ch=self.data["channels"],
            weights=None,
            verbose=verbose and RANK in {-1, 0},
        )
        if weights is not None:
            model.load(weights)
        return model

    def get_validator(self):
        self.loss_names = "box_loss", "cls_loss", "dfl_loss", "val_loss"
        return ValueValidator(
            self.test_loader, save_dir=self.save_dir, args=copy(self.args), _callbacks=self.callbacks
        )

    def set_class_weights(self):
        if not (0 < self.args.cls_pw <= 1.0):
            return
        combined = np.concatenate([lb["cls"].flatten() for lb in self.train_loader.dataset.labels], 0)
        types = (combined.astype(int) // VALUE_STRIDE)
        counts = np.bincount(types, minlength=self.data["nc"]).astype(np.float32)
        counts = np.where(counts == 0, 1.0, counts)
        weights = (1.0 / counts) ** self.args.cls_pw
        weights = weights / weights.mean()
        self.model.class_weights = torch.from_numpy(weights).to(self.device)


def train_value_model(
    data: str,
    model_cfg: str = "yolo11n.yaml",
    weights: str | None = None,
    nv: int = NUM_VALUE_CLASSES,
    **train_args: Any,
) -> ValueTrainer:
    """Train the dual-head dice model

    Args:
        data: Path to the dataset YAML (``nc: 7`` with the 7 dice-type names)
        model_cfg: Base YOLO model YAML (e.g. ``yolo11n.yaml``)
        weights: Optional pretrained ``.pt`` to warm-start backbone + box/type head
        nv: Number of value classes (21)
        **train_args: Standard ultralytics training overrides (epochs, imgsz, batch, device, ...)

    Returns:
        The fitted :class:`ValueTrainer`
    """
    patch_verify_image_label()

    trainer_cls = type("ConfiguredValueTrainer", (ValueTrainer,), {"nv": nv})
    overrides: dict[str, Any] = {"model": weights or model_cfg, "data": data}
    overrides.update(train_args)
    trainer = trainer_cls(overrides=overrides)
    trainer.train()
    return trainer


In [ ]:
import json
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from IPython.display import display
from PIL import Image as PILImage, ImageDraw, ImageFont

KAGGLE_DATASET_DIR = Path('/kaggle/input/datasets/oskarwichtowski/d-and-d-dices-synthetic-dataset')
WORK_DIR = Path('/kaggle/working')
YOLO_DATA_DIR = WORK_DIR / 'yolo_dataset'
RUNS_DIR = WORK_DIR / 'runs'

EPOCHS = 150
BATCH = 16
IMGSZ = 640
BASE_MODEL = 'yolo11n.yaml'
PRETRAINED = 'yolo11n.pt'

DICE_VALUE_RANGES = {
    'D4':   list(range(1, 5)),
    'D6':   list(range(1, 7)),
    'D8':   list(range(1, 9)),
    'D10':  list(range(0, 10)),
    'D12':  list(range(1, 13)),
    'D20':  list(range(1, 21)),
    'D100': list(range(0, 100, 10)),
}

TRAIN_RATIO, VAL_RATIO = 0.80, 0.15
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Types:', NUM_TYPE_CLASSES, '| value classes:', NUM_VALUE_CLASSES)
print(f'Dataset: {KAGGLE_DATASET_DIR}  exists={KAGGLE_DATASET_DIR.exists()}')


## 2. Prepare Dataset (combined type+value labels)

Each YOLO label line is `combined_id x y w h`, where `combined_id = type*100 + value_class`.
`data.yaml` declares the 7 dice types; the value is carried in the combined id and
supervised by the custom loss.


In [ ]:
def extract_render_group(filename: str) -> str:
    m = re.search(r'render_(\d+)', filename)
    return str(int(m.group(1)) // 100) if m else filename

img_dir = KAGGLE_DATASET_DIR / 'images'
ann_dir = KAGGLE_DATASET_DIR / 'annotations'
ann_files = sorted(ann_dir.glob('*.json'))
pairs = [(img_dir / f'{a.stem}.png', a) for a in ann_files if (img_dir / f'{a.stem}.png').exists()]
print(f'{len(pairs)} image+annotation pairs')

groups = defaultdict(list)
for pair in pairs:
    groups[extract_render_group(pair[0].stem)].append(pair)
group_keys = list(groups.keys())
random.shuffle(group_keys)
n = len(group_keys)
n_train, n_val = int(n * TRAIN_RATIO), int(n * VAL_RATIO)
split_groups = {
    'train': group_keys[:n_train],
    'val': group_keys[n_train:n_train + n_val],
    'test': group_keys[n_train + n_val:],
}
splits = {k: [p for g in gs for p in groups[g]] for k, gs in split_groups.items()}
for sp in splits.values():
    random.shuffle(sp)
print('Split:', {k: len(v) for k, v in splits.items()})


In [ ]:
if YOLO_DATA_DIR.exists():
    shutil.rmtree(YOLO_DATA_DIR)
for _split in ['train', 'val', 'test']:
    (YOLO_DATA_DIR / 'images' / _split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / 'labels' / _split).mkdir(parents=True, exist_ok=True)

type_counts = Counter()
value_counts = defaultdict(Counter)
skipped_images = 0
for split_name, split_pairs in splits.items():
    img_out = YOLO_DATA_DIR / 'images' / split_name
    lbl_out = YOLO_DATA_DIR / 'labels' / split_name
    for img_path, ann_path in split_pairs:
        lines = convert_annotation(ann_path)
        if not lines:
            skipped_images += 1
            continue
        dst = img_out / img_path.name
        if not dst.exists():
            dst.symlink_to(img_path)
        (lbl_out / f'{img_path.stem}.txt').write_text('\n'.join(lines) + '\n')
        for line in lines:
            tid, vclass = decode_combined(int(line.split()[0]))
            dtype = CLASS_TO_DICE_TYPE[tid]
            type_counts[dtype] += 1
            value_counts[dtype][vclass] += 1
print(f'Skipped {skipped_images} images with no valid dice')
print('Per-type counts:', dict(type_counts))


In [ ]:
data_yaml = {
    'path': str(YOLO_DATA_DIR),
    'train': str(YOLO_DATA_DIR / 'images' / 'train'),
    'val': str(YOLO_DATA_DIR / 'images' / 'val'),
    'test': str(YOLO_DATA_DIR / 'images' / 'test'),
    'nc': NUM_TYPE_CLASSES,
    'names': {i: name for name, i in DICE_TYPE_TO_CLASS.items()},
}
data_yaml_path = YOLO_DATA_DIR / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print(yaml.safe_dump(data_yaml, default_flow_style=False))


## 3. Sanity Check — Visualize a Few Samples


In [ ]:
TYPE_COLORS = {
    'D4': '#FF6B6B', 'D6': '#4ECDC4', 'D8': '#45B7D1', 'D10': '#96CEB4',
    'D12': '#FFEAA7', 'D20': '#DDA0DD', 'D100': '#98D8C8',
}

def _font(size=14):
    try:
        return ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', size)
    except OSError:
        return ImageFont.load_default()

train_imgs = list((YOLO_DATA_DIR / 'images' / 'train').glob('*.png'))
for img_path in random.sample(train_imgs, min(3, len(train_imgs))):
    label_path = YOLO_DATA_DIR / 'labels' / 'train' / f'{img_path.stem}.txt'
    img = PILImage.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img); w, h = img.size; font = _font()
    for line in label_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        combined, xc, yc, bw, bh = line.split()
        tid, vclass = decode_combined(int(combined))
        dtype = CLASS_TO_DICE_TYPE[tid]; value = class_to_value(dtype, vclass)
        xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
        x1 = int((xc - bw / 2) * w); y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w); y2 = int((yc + bh / 2) * h)
        color = TYPE_COLORS.get(dtype, '#FFFFFF')
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1, y1 - 16), f'{dtype}:{value}', fill=color, font=font)
    img.thumbnail((512, 512))
    display(img)


## 4. Train Dual-Head Model

4-component loss: `box`, `cls` (type), `dfl`, `val` (value). Flips disabled to preserve
value orientation.


In [ ]:
CHECKPOINT_DIR = RUNS_DIR / 'detection' / 'dice_detector' / 'weights'
RESUME_PATH = CHECKPOINT_DIR / 'last.pt'
resume = RESUME_PATH.exists()
warm_start = None if resume else PRETRAINED
print('Resuming' if resume else f'Fresh start (warm-start: {warm_start})')

trainer = train_value_model(
    data=str(data_yaml_path),
    model_cfg=BASE_MODEL,
    weights=str(RESUME_PATH) if resume else warm_start,
    nv=NUM_VALUE_CLASSES,
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=4,
    project=str(RUNS_DIR / 'detection'), name='dice_detector',
    exist_ok=True, resume=resume, patience=30, save=True, save_period=10,
    plots=True, verbose=True,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
    degrees=15.0, translate=0.1, scale=0.5,
    flipud=0.0, fliplr=0.0, mosaic=1.0, mixup=0.1, multi_scale=True,
)
results_dir = RUNS_DIR / 'detection' / 'dice_detector'
print('Done. Results:', results_dir)


## 4b. Evaluate Value Head Accuracy

The validator only reports type-level detection mAP. This cell runs the trained model
with `export_value=True` on the **test split** and measures how often the value head
picks the correct face value — the metric that actually matters for a dice reader.

In [ ]:
# --- Value head evaluation on test set ---
import cv2
from ultralytics.utils.ops import non_max_suppression

best_weights = results_dir / 'weights' / 'best.pt'
eval_model = build_value_model(
    cfg=BASE_MODEL, nc=NUM_TYPE_CLASSES, nv=NUM_VALUE_CLASSES, weights=str(best_weights)
)
eval_model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
device = next(eval_model.parameters()).device

eval_model.model[-1].export_value = True

test_img_dir = YOLO_DATA_DIR / 'images' / 'test'
test_lbl_dir = YOLO_DATA_DIR / 'labels' / 'test'
test_images = sorted(test_img_dir.glob('*.png'))
print(f'Test images: {len(test_images)}')

# IoU threshold for matching predictions to ground truth
IOU_THRESH = 0.5

type_correct = 0
type_total = 0
value_correct = 0
value_total = 0
per_type_value_correct = defaultdict(int)
per_type_value_total = defaultdict(int)
value_confusion = []  # (dice_type, gt_value, pred_value)

for img_path in test_images:
    # Load ground truth
    lbl_path = test_lbl_dir / f'{img_path.stem}.txt'
    if not lbl_path.exists():
        continue
    gt_boxes = []
    for line in lbl_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        combined_id = int(parts[0])
        gt_tid, gt_vclass = decode_combined(combined_id)
        xc, yc, bw, bh = map(float, parts[1:5])
        gt_boxes.append((gt_tid, gt_vclass, xc, yc, bw, bh))
    if not gt_boxes:
        continue

    # Preprocess image
    img = cv2.imread(str(img_path))
    h0, w0 = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Letterbox to IMGSZ
    r = min(IMGSZ / h0, IMGSZ / w0)
    new_h, new_w = int(h0 * r), int(w0 * r)
    img_resized = cv2.resize(img_rgb, (new_w, new_h))
    canvas = np.full((IMGSZ, IMGSZ, 3), 114, dtype=np.uint8)
    dh, dw = (IMGSZ - new_h) // 2, (IMGSZ - new_w) // 2
    canvas[dh:dh + new_h, dw:dw + new_w] = img_resized

    tensor = torch.from_numpy(canvas).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0

    # Forward pass
    with torch.no_grad():
        preds = eval_model(tensor)

    # preds shape: (1, 4+nc+nv, num_anchors) — box(4) + type_scores(7) + value_scores(21)
    if isinstance(preds, (list, tuple)):
        preds = preds[0]


    p = preds.permute(0, 2, 1)  # (1, anchors, 4+nc+nv)
    det_part = p[..., :4 + NUM_TYPE_CLASSES]  # (1, anchors, 4+7)

    # Manual NMS on the detection part
    nms_results = non_max_suppression(det_part.permute(0, 2, 1), conf_thres=0.25, iou_thres=0.45)

    if len(nms_results) == 0 or len(nms_results[0]) == 0:
        continue

    dets = nms_results[0]  # (N, 6): x1,y1,x2,y2,conf,cls

    # Get the value scores for matched anchor indices
    # We need to match NMS output back to the value scores
    # Since NMS changes indices, we'll use the detection boxes to match GT

    for det in dets:
        x1, y1, x2, y2, conf, cls_id = det.cpu().numpy()
        pred_tid = int(cls_id)

        # Convert detection box back to normalized coords (undo letterbox)
        px1 = (x1 - dw) / (w0 * r)
        py1 = (y1 - dh) / (h0 * r)
        px2 = (x2 - dw) / (w0 * r)
        py2 = (y2 - dh) / (h0 * r)
        pred_xc = (px1 + px2) / 2
        pred_yc = (py1 + py2) / 2
        pred_w = px2 - px1
        pred_h = py2 - py1

        # Find best matching GT box by IoU
        best_iou = 0
        best_gt = None
        for gt_tid, gt_vclass, gxc, gyc, gbw, gbh in gt_boxes:
            # Compute IoU between pred and GT (both in normalized coords)
            gx1, gy1 = gxc - gbw/2, gyc - gbh/2
            gx2, gy2 = gxc + gbw/2, gyc + gbh/2
            ix1 = max(px1, gx1); iy1 = max(py1, gy1)
            ix2 = min(px2, gx2); iy2 = min(py2, gy2)
            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
            area_p = pred_w * pred_h
            area_g = gbw * gbh
            union = area_p + area_g - inter
            iou = inter / union if union > 0 else 0
            if iou > best_iou:
                best_iou = iou
                best_gt = (gt_tid, gt_vclass)

        if best_iou < IOU_THRESH or best_gt is None:
            continue

        gt_tid, gt_vclass = best_gt
        type_total += 1
        if pred_tid == gt_tid:
            type_correct += 1

        # Get value prediction from the full prediction tensor
        # Find the anchor closest to our detection
        anchors = p[0]  # (anchors, 4+nc+nv)
        value_scores = anchors[:, 4 + NUM_TYPE_CLASSES:]  # (anchors, nv)

        # Find anchor whose box center is closest to det center
        anchor_boxes = anchors[:, :4]  # (anchors, 4) in xywh or xyxy
        det_center = torch.tensor([(x1+x2)/2, (y1+y2)/2], device=device)
        anchor_centers = (anchor_boxes[:, :2] + anchor_boxes[:, 2:4]) / 2  # rough center
        dists = ((anchor_centers - det_center) ** 2).sum(dim=1)
        nearest = dists.argmin()
        pred_vclass = int(value_scores[nearest].argmax())

        value_total += 1
        dtype_name = CLASS_TO_DICE_TYPE[gt_tid]
        gt_value = class_to_value(dtype_name, gt_vclass)
        pred_value = class_to_value(dtype_name, pred_vclass)

        per_type_value_total[dtype_name] += 1
        if pred_vclass == gt_vclass:
            value_correct += 1
            per_type_value_correct[dtype_name] += 1
        else:
            value_confusion.append((dtype_name, gt_value, pred_value))

# Disable export mode
eval_model.model[-1].export_value = False

print(f"\n{'='*60}")
print(f"TEST SET EVALUATION (value head)")
print(f"{'='*60}")
print(f"Type accuracy:  {type_correct}/{type_total} = {type_correct/type_total:.1%}" if type_total else "No detections")
print(f"Value accuracy: {value_correct}/{value_total} = {value_correct/value_total:.1%}" if value_total else "No value preds")

print(f"\nPer-type value accuracy:")
for dtype in sorted(per_type_value_total):
    c = per_type_value_correct[dtype]
    n = per_type_value_total[dtype]
    print(f"  {dtype:>4s}: {c}/{n} = {c/n:.1%}")

# Show worst offenders
if value_confusion:
    print(f"\nSample misclassifications (first 20):")
    for dtype, gt_v, pred_v in value_confusion[:20]:
        print(f"  {dtype}: GT={gt_v} → Pred={pred_v}")

value_eval_results = {
    'type_accuracy': type_correct / type_total if type_total else 0,
    'value_accuracy': value_correct / value_total if value_total else 0,
    'type_total': type_total,
    'value_total': value_total,
    'per_type': {
        dtype: {
            'correct': per_type_value_correct[dtype],
            'total': per_type_value_total[dtype],
            'accuracy': per_type_value_correct[dtype] / per_type_value_total[dtype]
        }
        for dtype in sorted(per_type_value_total)
    },
}
print(f"\nResults saved as 'value_eval_results'")

## 5. Export for Local Validation

Bundle `best.pt`, `last.pt`, `data.yaml` and training plots into a single folder to
download. Validate locally with `notebooks/local/validate.ipynb`.


In [ ]:
export_dir = WORK_DIR / 'dice_detector_export'
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

weights_dir = results_dir / 'weights'
for f in weights_dir.glob('*.pt'):
    shutil.copy2(f, export_dir / f.name)
for plot in results_dir.glob('*.png'):
    shutil.copy2(plot, export_dir / plot.name)
if (results_dir / 'results.csv').exists():
    shutil.copy2(results_dir / 'results.csv', export_dir / 'results.csv')
shutil.copy2(data_yaml_path, export_dir / 'data.yaml')

# Note the config needed to rebuild the custom head locally.
(export_dir / 'model_config.json').write_text(json.dumps({
    'base_model': BASE_MODEL,
    'nc': NUM_TYPE_CLASSES,
    'nv': NUM_VALUE_CLASSES,
    'imgsz': IMGSZ,
}, indent=2))

# Save value evaluation results
if value_eval_results:
    with open(export_dir / 'value_eval_results.json', 'w') as f:
        json.dump(value_eval_results, f, indent=2)

print('Exported to', export_dir)
for f in sorted(export_dir.iterdir()):
    print(f'  {f.name}: {f.stat().st_size / 1024:.1f} KB')


## 6. Cross-Domain Validation (Real-World Dice)

Validate the synthetic-trained detector against **real photographs** from the
[ucffool/dice-d4-d6-d8-d10-d12-d20-images](https://www.kaggle.com/datasets/ucffool/dice-d4-d6-d8-d10-d12-d20-images)
dataset (CC0, ~16 K images, 6 types, no D100).

This measures the **synthetic→real domain gap** for dice type classification.

**Setup:** Add `ucffool/dice-d4-d6-d8-d10-d12-d20-images` as a second input
dataset in the Kaggle notebook settings. It will appear under
`/kaggle/input/dice-d4-d6-d8-d10-d12-d20-images/dice/`.

In [ ]:
# --- Cross-domain validation against ucffool real-world dice photos ---
UCFFOOL_DIR = Path('/kaggle/input/datasets/ucffool/dice-d4-d6-d8-d10-d12-d20-images')

# Map ucffool folder names → our YOLO type class IDs
UCFFOOL_TYPE_MAP = {
    "d4": 0,   # D4
    "d6": 1,   # D6
    "d8": 2,   # D8
    "d10": 3,  # D10
    "d12": 4,  # D12
    "d20": 5,  # D20
}

if not UCFFOOL_DIR.exists():
    print(f"⚠ ucffool dataset not found at {UCFFOOL_DIR}")
    print("  Add 'ucffool/dice-d4-d6-d8-d10-d12-d20-images' as a second input dataset.")
    ucffool_results = None
else:
    from ultralytics import YOLO as _YOLO

    # Load the best model from this training run
    best_path = results_dir / 'weights' / 'best.pt'
    cross_model = _YOLO(str(best_path))

    # Collect all images from train/ and valid/ splits
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    ucffool_images = []
    for split in ('train', 'valid'):
        split_dir = UCFFOOL_DIR / split
        if not split_dir.exists():
            continue
        for type_dir in sorted(split_dir.iterdir()):
            if not type_dir.is_dir():
                continue
            folder_name = type_dir.name.lower()
            if folder_name not in UCFFOOL_TYPE_MAP:
                continue
            gt_type_id = UCFFOOL_TYPE_MAP[folder_name]
            for img_path in sorted(type_dir.iterdir()):
                if img_path.suffix.lower() in image_exts:
                    ucffool_images.append((img_path, gt_type_id, folder_name))

    print(f"Found {len(ucffool_images)} real-world images across {len(UCFFOOL_TYPE_MAP)} types")
    type_counts = Counter(folder for _, _, folder in ucffool_images)
    for t in sorted(type_counts):
        print(f"  {t}: {type_counts[t]} images")

    # Run detection on each image (single die per image).
    # The dual-head model's detect output uses combined class IDs
    # (type * VALUE_STRIDE + value_class), so we decode to extract the type.
    correct = 0
    total = 0
    per_type_correct = defaultdict(int)
    per_type_total = defaultdict(int)
    confusion = defaultdict(lambda: defaultdict(int))
    no_detection = 0

    CROSS_BATCH = 32
    for batch_start in range(0, len(ucffool_images), CROSS_BATCH):
        batch = ucffool_images[batch_start : batch_start + CROSS_BATCH]
        batch_paths = [str(p) for p, _, _ in batch]
        batch_results = cross_model(batch_paths, imgsz=IMGSZ, verbose=False)

        for (img_path, gt_type_id, gt_name), result in zip(batch, batch_results):
            total += 1
            per_type_total[gt_name] += 1

            if len(result.boxes) == 0:
                no_detection += 1
                confusion[gt_name]['<none>'] += 1
                continue

            # Take highest-confidence detection
            best_idx = result.boxes.conf.argmax()
            pred_cls = int(result.boxes.cls[best_idx])

            # The model nc may be 7 (plain type IDs) or 700 (combined IDs).
            # If combined, decode; otherwise use directly.
            if pred_cls >= NUM_TYPE_CLASSES:
                pred_type_id, _ = decode_combined(pred_cls)
            else:
                pred_type_id = pred_cls
            pred_name = CLASS_TO_DICE_TYPE.get(pred_type_id, f'cls{pred_type_id}').lower()

            confusion[gt_name][pred_name] += 1
            if pred_type_id == gt_type_id:
                correct += 1
                per_type_correct[gt_name] += 1

    accuracy = correct / total if total > 0 else 0
    print(f"\n{'='*60}")
    print(f"Cross-Domain Type Accuracy: {correct}/{total} = {accuracy:.1%}")
    print(f"No detections: {no_detection}/{total} = {no_detection/total:.1%}")
    print(f"{'='*60}")

    print('\nPer-type accuracy:')
    for t in sorted(per_type_total):
        c = per_type_correct[t]
        n = per_type_total[t]
        print(f"  {t:>4s}: {c}/{n} = {c/n:.1%}")

    # Mini confusion matrix
    all_types = sorted(set(list(confusion.keys()) + [
        p for row in confusion.values() for p in row.keys()
    ]))
    print(f'\nConfusion matrix (rows=GT, cols=predicted):')
    header = '        ' + ' '.join(f'{t:>6s}' for t in all_types)
    print(header)
    for gt in sorted(confusion):
        row = ' '.join(f'{confusion[gt].get(t, 0):>6d}' for t in all_types)
        print(f'  {gt:>4s}  {row}')

    ucffool_results = {
        'total_images': total,
        'correct': correct,
        'accuracy': accuracy,
        'no_detections': no_detection,
        'per_type': {
            t: {'correct': per_type_correct[t], 'total': per_type_total[t],
                'accuracy': per_type_correct[t] / per_type_total[t] if per_type_total[t] > 0 else 0}
            for t in sorted(per_type_total)
        },
    }
    print(f"\nResults dict saved as 'ucffool_results'")